# Hyperparameter Tuning — CNN-1D URL Classifier
**Google Colab | Keras Tuner HyperBand | Dataset 50.000 Balanced**

---
Notebook ini mencari kombinasi arsitektur dan hyperparameter terbaik untuk
model CNN-1D berbasis karakter untuk klasifikasi URL.

**Arsitektur yang dituning (Multi-scale CNN):**
```
Input [MAX_LEN, int32]
  → Embedding(vocab=41, dim=?)
  → 3 Conv1D paralel (kernel 3, 5, 7) → GlobalMaxPool → Concatenate
  → Dense(?) → Dropout → Dense(?) → Sigmoid
```

**Hyperparameter yang dicari:**
- `embed_dim`: dimensi embedding karakter
- `num_filters`: jumlah filter per Conv1D
- `dense_units`: ukuran Dense layer
- `dropout_rate`: tingkat dropout
- `learning_rate`: laju belajar optimizer Adam

> Menggunakan **50K** data (lebih kecil) untuk mempercepat tuning.
> Training final dengan 200K ada di notebook `cnn1d_training.ipynb`.

## Cell 1 — Install Library

In [ ]:
# keras-tuner: library untuk hyperparameter search di Keras/TensorFlow
!pip install -q keras-tuner
!pip install -q --upgrade scikit-learn
print('Install selesai.')

## Cell 2 — Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import re
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

print(f'TensorFlow   : {tf.__version__}')
print(f'Keras Tuner  : {kt.__version__}')
# Pastikan GPU tersedia di Colab (Runtime → Change runtime type → T4 GPU)
print(f'GPU tersedia : {len(tf.config.list_physical_devices("GPU")) > 0}')

## Cell 3 — Mount Google Drive & Konfigurasi

**Format dataset yang diharapkan** — CSV dengan kolom:
```
url, label   (atau)   domain, label
```
> `label`: 0 = URL aman, 1 = URL pornografi

Jika dataset sudah memiliki kolom fitur RF (`domain_length`, dst.),
notebook ini akan menggunakan kolom `url` untuk CNN-1D.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ================================================================
# SESUAIKAN PATH INI
# ================================================================
DATASET_PATH = '/content/drive/MyDrive/Tugas Akhir/Dataset/dataset_url.csv'
SAVE_PATH    = '/content/drive/MyDrive/Tugas Akhir/Model/CNN1D/'
URL_COL      = 'url'    # Nama kolom URL atau domain di dataset
LABEL_COL    = 'label'
# ================================================================

os.makedirs(SAVE_PATH, exist_ok=True)

df = pd.read_csv(DATASET_PATH)
print(f'Dataset: {df.shape[0]:,} baris x {df.shape[1]} kolom')
print(f'Distribusi label:\n{df[LABEL_COL].value_counts()}')
df[[URL_COL, LABEL_COL]].head()

## Cell 4 — Fungsi Tokenisasi

> **PENTING**: Semua fungsi ini HARUS IDENTIK dengan kode di `UrlClassifier.kt`.

**CHAR_TO_IDX** — pemetaan karakter ke indeks:
```
0      → PAD (karakter kosong)
1      → UNK (karakter tidak dikenal)
2–27   → a–z
28–37  → 0–9
38     → titik (.)
39     → hubung (-)
40     → garis bawah (_)
```
Total vocab: 41 karakter.

In [ ]:
# Bangun CHAR_TO_IDX — identik dengan Android UrlClassifier.kt
CHAR_TO_IDX = {chr(0): 0}  # PAD = indeks 0
for i, c in enumerate('abcdefghijklmnopqrstuvwxyz'):
    CHAR_TO_IDX[c] = i + 2       # a=2, b=3, ..., z=27
for i, c in enumerate('0123456789'):
    CHAR_TO_IDX[c] = i + 28      # 0=28, 1=29, ..., 9=37
CHAR_TO_IDX['.'] = 38
CHAR_TO_IDX['-'] = 39
CHAR_TO_IDX['_'] = 40

UNK_IDX    = 1
PAD_IDX    = 0
VOCAB_SIZE = 41  # indeks 0–40

# TLD yang dikenal (dipakai untuk ekstrak core domain)
COMMON_TLDS = {
    'com','net','org','info','biz','co','io','me','tv','cc','in','ru','cn',
    'jp','kr','de','uk','fr','it','es','br','au','ca','nl','id','my','th',
    'ph','vn','sg','hk','tw','xyz','top','site','online','club','live',
    'fun','space','tech','store','shop','app','dev','porn','sex','xxx',
    'adult','cam','tube','tk','ml','ga','cf','gq','vip','pw'
}
SECOND_LEVEL_TLDS = {
    'co.id','co.uk','co.jp','co.kr','com.au','com.br','com.cn','com.hk',
    'com.my','com.sg','com.tw','com.vn','net.id','ac.id','go.id','web.id'
}

def normalize_domain(url: str) -> str:
    """Normalisasi URL ke domain — identik dengan Android normalizeDomain()"""
    url = url.lower().strip()
    for prefix in ['https://', 'http://', 'www.']:
        if url.startswith(prefix):
            url = url[len(prefix):]
    url = url.split('/')[0].split('?')[0].split('#')[0].split(':')[0]
    return url

def extract_core_domain(full_domain: str) -> str:
    """Ekstrak nama inti domain — identik dengan Android extractMainDomainName()"""
    parts = full_domain.split('.')
    if len(parts) < 2:
        return full_domain
    if len(parts) >= 3:
        potential_2nd = f'{parts[-2]}.{parts[-1]}'
        if potential_2nd in SECOND_LEVEL_TLDS:
            return parts[-3]
    if parts[-1] in COMMON_TLDS:
        return parts[-2]
    return parts[-2] if len(parts) >= 2 else full_domain

def tokenize(domain: str, max_len: int) -> list:
    """Ubah string domain menjadi array indeks karakter — identik dengan Android tokenize()"""
    tokens = [PAD_IDX] * max_len
    for i, char in enumerate(domain[:max_len]):
        tokens[i] = CHAR_TO_IDX.get(char, UNK_IDX)
    return tokens

# Verifikasi
test_urls = [
    'https://www.youporn.com/watch/123',
    'google.com',
    'xvideos.com',
]
print(f'{"URL":<45} {"Domain":<20} {"Core"}')
print('-' * 75)
for url in test_urls:
    d  = normalize_domain(url)
    core = extract_core_domain(d)
    print(f'{url:<45} {d:<20} {core}')
    print(f'  Token[0:10]: {tokenize(core, 34)[:10]}...')

## Cell 5 — Siapkan Dataset 50K untuk Tuning

Tuning menggunakan **50K sampel** (25K per kelas) agar lebih cepat.
Training final tetap menggunakan 200K.

**Domain Augmentation:**
Setiap URL menghasilkan **2 sampel**:
1. Domain penuh: `youporn.com`
2. Domain inti: `youporn`

Ini mengajarkan model untuk mengenali URL porno baik dengan maupun tanpa TLD.

In [ ]:
N_TUNE_PER_CLASS = 25_000  # 25K aman + 25K porno = 50K total

df_safe = df[df[LABEL_COL] == 0].sample(n=N_TUNE_PER_CLASS, random_state=42)
df_porn = df[df[LABEL_COL] == 1].sample(n=N_TUNE_PER_CLASS, random_state=42)
df_tune = pd.concat([df_safe, df_porn]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset tuning: {len(df_tune):,} baris')
print('Membangun sekuens karakter dengan augmentasi domain...')

# Hitung MAX_LEN dari persentil 95 panjang domain
all_domains = df_tune[URL_COL].apply(normalize_domain)
domain_lengths = all_domains.apply(len)
MAX_LEN = int(np.percentile(domain_lengths, 95))
MAX_LEN = max(MAX_LEN, 20)  # Minimal 20 karakter
MAX_LEN = min(MAX_LEN, 100) # Maksimal 100 karakter

print(f'\nAnalisis panjang domain:')
print(f'  Min    : {domain_lengths.min()}')
print(f'  Median : {domain_lengths.median():.0f}')
print(f'  P95    : {np.percentile(domain_lengths, 95):.0f}')
print(f'  Max    : {domain_lengths.max()}')
print(f'  MAX_LEN yang dipakai: {MAX_LEN}')

# Bangun sequences dengan augmentasi
X_list, y_list = [], []

for _, row in df_tune.iterrows():
    label = int(row[LABEL_COL])
    full_domain = normalize_domain(str(row[URL_COL]))
    core_domain = extract_core_domain(full_domain)

    if not full_domain or '.' not in full_domain:
        continue

    # Sampel 1: domain penuh (misal: youporn.com)
    X_list.append(tokenize(full_domain, MAX_LEN))
    y_list.append(label)

    # Sampel 2: domain inti (misal: youporn) — augmentasi
    if core_domain and core_domain != full_domain:
        X_list.append(tokenize(core_domain, MAX_LEN))
        y_list.append(label)

X = np.array(X_list, dtype=np.int32)
y = np.array(y_list, dtype=np.float32)

print(f'\nTotal sampel (setelah augmentasi): {len(X):,}')
print(f'Shape X: {X.shape}  (sampel × karakter)')
print(f'Distribusi: aman={int((y==0).sum()):,}, porno={int((y==1).sum()):,}')

# Split 80/20
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {len(X_train):,} | Val: {len(X_val):,}')

## Cell 6 — Definisi Model untuk Keras Tuner

Fungsi `build_model(hp)` mendefinisikan arsitektur CNN-1D yang bisa dikonfigurasi.
Keras Tuner akan memanggil fungsi ini berkali-kali dengan kombinasi `hp` yang berbeda.

In [ ]:
def build_model(hp):
    """
    Builder untuk CNN-1D Multi-scale.
    hp: objek HyperParameters dari Keras Tuner.
    """
    # ---- Hyperparameter yang akan dicari ----
    embed_dim     = hp.Choice('embed_dim',     [16, 32, 64])
    num_filters   = hp.Choice('num_filters',   [32, 64, 128])
    dense_units   = hp.Choice('dense_units',   [64, 128, 256])
    dropout_rate  = hp.Choice('dropout_rate',  [0.2, 0.3, 0.5])
    learning_rate = hp.Choice('learning_rate', [1e-2, 1e-3, 5e-4, 1e-4])

    # ---- Arsitektur Multi-scale CNN ----

    # Input: array indeks karakter [batch, MAX_LEN]
    inputs = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name='input')

    # Embedding: ubah indeks integer menjadi vektor padat
    # mask_zero=False karena PAD (idx=0) masih valid karakter
    x = tf.keras.layers.Embedding(
        input_dim=VOCAB_SIZE, output_dim=embed_dim,
        embeddings_initializer='uniform', name='embedding'
    )(inputs)

    # Multi-scale branches: menangkap pola n-gram panjang 3, 5, dan 7 karakter
    # Contoh: kernel=3 menangkap 'xxx', 'sex', 'cam'
    #         kernel=7 menangkap 'youporn', 'xvideos'
    branches = []
    for kernel_size in [3, 5, 7]:
        branch = tf.keras.layers.Conv1D(
            filters=num_filters, kernel_size=kernel_size,
            activation='relu', padding='same',
            name=f'conv_k{kernel_size}'
        )(x)
        # GlobalMaxPool: ambil aktivasi tertinggi sepanjang sequence
        # Membuat model position-independent (mendeteksi kata kunci di mana saja)
        branch = tf.keras.layers.GlobalMaxPooling1D(name=f'pool_k{kernel_size}')(branch)
        branches.append(branch)

    # Gabungkan output ketiga branch
    z = tf.keras.layers.Concatenate(name='concat')(branches)

    # Dense layers untuk klasifikasi akhir
    z = tf.keras.layers.Dense(dense_units, activation='relu', name='dense_1')(z)
    z = tf.keras.layers.Dropout(dropout_rate, name='drop_1')(z)
    z = tf.keras.layers.Dense(dense_units // 2, activation='relu', name='dense_2')(z)
    z = tf.keras.layers.Dropout(dropout_rate, name='drop_2')(z)

    # Output: probabilitas URL porno [0.0 - 1.0]
    output = tf.keras.layers.Dense(1, activation='sigmoid', name='output')(z)

    model = tf.keras.Model(inputs=inputs, outputs=output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),          # Area Under ROC curve
            tf.keras.metrics.Precision(name='prec'),   # Precision
            tf.keras.metrics.Recall(name='rec'),       # Recall
        ]
    )
    return model

# Preview model dengan parameter default
hp_preview = kt.HyperParameters()
hp_preview.Fixed('embed_dim', 32)
hp_preview.Fixed('num_filters', 64)
hp_preview.Fixed('dense_units', 128)
hp_preview.Fixed('dropout_rate', 0.3)
hp_preview.Fixed('learning_rate', 1e-3)

preview_model = build_model(hp_preview)
preview_model.summary()
print(f'\nTotal parameter: {preview_model.count_params():,}')

## Cell 7 — Jalankan Keras Tuner (HyperBand)

**HyperBand** bekerja dengan cara:
1. Coba banyak konfigurasi dengan epoch sedikit
2. Eliminasi konfigurasi buruk lebih awal
3. Fokuskan resource ke konfigurasi terbaik

Lebih efisien dari RandomSearch untuk model deep learning.

> Estimasi waktu: **20–60 menit** (tergantung GPU Colab).

In [ ]:
# HyperBand tuner
# objective='val_auc': optimalkan AUC di validation set
# max_epochs: maksimal epoch per konfigurasi
# factor: faktor eliminasi (1/3 yang tersisa di setiap ronde)
tuner = kt.Hyperband(
    hypermodel    = build_model,
    objective     = kt.Objective('val_auc', direction='max'),
    max_epochs    = 10,
    factor        = 3,
    directory     = '/content/kt_cnn1d',
    project_name  = 'cnn1d_url_classifier',
    overwrite     = True
)

print('Ruang pencarian:')
tuner.search_space_summary()

# Callback: hentikan lebih awal jika tidak ada kemajuan
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', patience=3, mode='max', restore_best_weights=True
)

print('\nMemulai tuning... (bisa memakan waktu 20–60 menit)')
t0 = time.time()

tuner.search(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

print(f'\nTuning selesai dalam {(time.time()-t0)/60:.1f} menit')

## Cell 8 — Hasil Tuning

In [ ]:
# Tampilkan 5 konfigurasi terbaik
print('Top 5 konfigurasi terbaik:')
print('=' * 60)
for i, trial in enumerate(tuner.oracle.get_best_trials(num_trials=5)):
    val_auc = trial.metrics.get_best_value('val_auc')
    print(f'\nRank {i+1} — val_AUC: {val_auc:.4f}')
    for k, v in trial.hyperparameters.values.items():
        print(f'  {k:<20}: {v}')

# Hyperparameter terbaik
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print('\n=== HYPERPARAMETER TERBAIK ===')
best_params = {
    'embed_dim'    : best_hp.get('embed_dim'),
    'num_filters'  : best_hp.get('num_filters'),
    'dense_units'  : best_hp.get('dense_units'),
    'dropout_rate' : best_hp.get('dropout_rate'),
    'learning_rate': best_hp.get('learning_rate'),
    'max_len'      : MAX_LEN,
    'vocab_size'   : VOCAB_SIZE,
}
print(json.dumps(best_params, indent=2))

## Cell 9 — Evaluasi Model Terbaik pada Validation Set

In [ ]:
# Build dan latih ulang model terbaik dengan lebih banyak epoch
best_model = tuner.get_best_models(num_models=1)[0]

# Evaluasi
y_pred_prob = best_model.predict(X_val, verbose=0).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)
y_val_int = y_val.astype(int)

print('Evaluasi pada Validation Set (20%):')
print('=' * 45)
print(f'  Accuracy  : {accuracy_score(y_val_int, y_pred):.4f}')
print(f'  F1-Score  : {f1_score(y_val_int, y_pred):.4f}')
print('=' * 45)
print(classification_report(y_val_int, y_pred,
      target_names=['Aman (0)', 'Pornografi (1)'], digits=4))

# Plot distribusi skor
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_pred_prob[y_val == 0], bins=50, alpha=0.6, label='URL Aman', color='green')
ax.hist(y_pred_prob[y_val == 1], bins=50, alpha=0.6, label='URL Porno', color='red')
ax.axvline(0.5, color='black', linestyle='--', label='Threshold 0.5')
ax.set_xlabel('Skor Prediksi (P_porno)')
ax.set_ylabel('Frekuensi')
ax.set_title('Distribusi Skor Prediksi — Best Model')
ax.legend()
plt.tight_layout()
plt.show()

## Cell 10 — Simpan Hyperparameter Terbaik ke JSON
File ini akan dibaca otomatis oleh notebook training.

In [ ]:
json_path = SAVE_PATH + 'best_params_cnn.json'
with open(json_path, 'w') as f:
    json.dump(best_params, f, indent=2)

print(f'✅ Tersimpan: {json_path}')
print('\nIsi file:')
print(json.dumps(best_params, indent=2))
print('\n--- SELESAI ---')
print('Langkah berikutnya: Buka notebook cnn1d_training.ipynb')